# MultiSpec-GeoDiff Stage-I Demo Notebook

This notebook walks through the Stage-I pipeline step by step:
1. Load and visualize an experimental NIST IR spectrum
2. Preprocess (interpolate, normalize, extract peaks)
3. Build coordinate-aware spectral embeddings
4. Apply spectral-distance-bias attention
5. Retrieve and rerank candidate molecules
6. Visualize Top-K results

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../03_code/src"))
import numpy as np
import matplotlib.pyplot as plt
from data_loader import load_ir_jsonl, split_query_library
from spectra_preprocess import preprocess_record
from spectra_encoder import CoordinateAwareSpectralEncoder
from candidate_generator import build_library_embeddings, retrieve_top_m
from reranker import rerank_candidates
from visualization import plot_spectrum_overlay, plot_attention_heatmap, plot_molecule_grid
from spectral_attention import compute_attention_heatmap
from metrics import topk_hit

In [ ]:
# 1. Load data
records = load_ir_jsonl("../04_data/IR_nist_200.jsonl")
print(f"Loaded {len(records)} records")

In [ ]:
# 2. Split query and library
query_id = 0
query_record, library_records = split_query_library(records, query_id)
print(f"Query SMILES: {query_record['smiles']}")
print(f"Library size: {len(library_records)}")

In [ ]:
# 3. Preprocess
query_processed = preprocess_record(query_record)
print(f"Processed: {len(query_processed['wavenumber'])} points, "
      f"{len(query_processed['peaks_wavenumber'])} peaks")

plt.figure(figsize=(10, 4))
plt.plot(query_processed['wavenumber'], query_processed['intensity'])
plt.scatter(query_processed['peaks_wavenumber'], query_processed['peaks_intensity'],
            color='red', s=15, label='Peaks')
plt.xlabel('Wavenumber (cm⁻¹)')
plt.ylabel('Normalized intensity')
plt.title(f'Query Spectrum: {query_record["smiles"]}')
plt.legend()
plt.grid(alpha=0.2)
plt.show()

In [ ]:
# 4. Encode
from spectra_preprocess import preprocess_records
library_processed = preprocess_records(library_records)

encoder = CoordinateAwareSpectralEncoder(hidden_dim=16)
query_embedding, _, attn = encoder.encode(query_processed)
print(f"Query embedding shape: {query_embedding.shape}")
print(f"Attention matrix shape: {attn.shape}")

In [ ]:
# 5. Attention heatmap
heatmap = compute_attention_heatmap(attn, downsample=8)
plt.figure(figsize=(6, 5))
plt.imshow(heatmap, aspect='auto', cmap='magma')
plt.colorbar(shrink=0.8)
plt.title('Spectral Distance Bias Attention')
plt.xlabel('Token index')
plt.ylabel('Token index')
plt.show()

In [ ]:
# 6. Retrieve
library_embeddings, library_smiles = build_library_embeddings(library_processed, encoder)
indices, ret_scores = retrieve_top_m(query_embedding, library_embeddings, top_m=10)
print(f"Retrieved {len(indices)} candidates")
print(f"Top-1 SMILES: {library_smiles[indices[0]]} (score: {ret_scores[0]:.4f})")

In [ ]:
# 7. Rerank
result_df = rerank_candidates(
    query_processed, library_processed, indices, ret_scores, top_k=5
)
result_df

In [ ]:
# 8. Check if ground truth is in top-k
hit = topk_hit(result_df['smiles'].tolist(), query_record['smiles'], k=5)
print(f"Ground truth in Top-5: {hit}")